# 08 — Feature Engineering Avancé

### Étape 7A — Analyse du potentiel prédictif

On évalue quelles familles de variables portent le signal prédictif, sans créer de nouvelles features pour l'instant.

In [ ]:
import warnings
import os

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Bibliothèques chargées.")

In [ ]:
TARGET = "fillRate_target_t_plus_1"
DATE_SPLIT = "2025-03-15"

data = pd.read_parquet(
    "../data/processed/orbit_dataset_engineering.parquet"
)
data["date_collecte"] = pd.to_datetime(data["date_collecte"])
data = data.sort_values("date_collecte").reset_index(drop=True)

print("Dataset chargé :", data.shape)

In [ ]:
# ============================================================
# ÉTAPE 7A — ANALYSE DU POTENTIEL PRÉDICTIF
# ============================================================


TARGET = "fillRate_target_t_plus_1"


# 1. Historique du remplissage
FEATURES_HISTORIQUE = [
    "fillRate",
    "fillRate_t_minus_1",
    "fillRate_t_minus_2",
    "delta_fillRate_24h",
    "rolling_mean_3d"
]


# 2. Variables météo
FEATURES_METEO = [
    "temperature_celsius",
    "humidite_pct",
    "indice_meteo"
]


# 3. Variables temporelles / calendrier
FEATURES_CALENDRIER = [
    "jour_semaine_num",
    "is_weekend",
    "is_jour_marche"
]


# Vérification
groupes = {
    "Historique": FEATURES_HISTORIQUE,
    "Météo": FEATURES_METEO,
    "Calendrier": FEATURES_CALENDRIER
}


for nom, features in groupes.items():
    disponibles = [f for f in features if f in data.columns]
    manquantes = [f for f in features if f not in data.columns]

    print(f"\n{nom}")
    print(f"  Disponibles : {len(disponibles)} / {len(features)}")

    if manquantes:
        print(f"  ⚠️ Manquantes : {manquantes}")
    else:
        print(f"  ✅ Toutes les variables sont disponibles")

In [ ]:
# ============================================================
# 4 CONFIGURATIONS DE FEATURES
# ============================================================


FEATURES_HISTORIQUE_METEO = (
    FEATURES_HISTORIQUE
    + FEATURES_METEO
)


FEATURES_HISTORIQUE_METEO_CALENDRIER = (
    FEATURES_HISTORIQUE
    + FEATURES_METEO
    + FEATURES_CALENDRIER
)


# Toutes les features utilisées par le pipeline corrigé
FEATURES_COMPLETES = [
    col for col in data.columns
    if col not in [
        "id_point",
        "date_collecte",
        TARGET,
        "statut_prevu",
        "priorite_recommandee",
        "priorite_prediction",
        "action_recommandee"
    ]
]


configs = {
    "Historique": FEATURES_HISTORIQUE,
    "Historique + météo": FEATURES_HISTORIQUE_METEO,
    "Historique + météo + calendrier": FEATURES_HISTORIQUE_METEO_CALENDRIER,
    "Toutes les features": FEATURES_COMPLETES
}


for nom, features in configs.items():
    print(f"{nom}: {len(features)} features")

In [ ]:
# ============================================================
# CONTRÔLE ANTI-FUITE
# ============================================================


VARIABLES_INTERDITES = {
    "statut_prevu",
    "priorite_recommandee",
    "priorite_prediction",
    "action_recommandee",
    TARGET
}


for nom, features in configs.items():
    fuite = set(features) & VARIABLES_INTERDITES

    if fuite:
        print(f"❌ {nom} : fuite détectée -> {fuite}")
    else:
        print(f"✅ {nom} : aucune variable interdite")

# ============================================================
# CELLULE 4 — SÉPARATION TEMPORELLE TRAIN / TEST
# ============================================================


# Vérification et préparation de la date
data["date_collecte"] = pd.to_datetime(data["date_collecte"])


# Même frontière temporelle que celle corrigée dans le pipeline
DATE_CUTOFF = pd.Timestamp("2025-03-15")


train_data = data[data["date_collecte"] < DATE_CUTOFF].copy()
test_data = data[data["date_collecte"] >= DATE_CUTOFF].copy()


print("SÉPARATION TEMPORELLE")
print("-" * 50)


print("TRAIN")
print("  Début :", train_data["date_collecte"].min())
print("  Fin   :", train_data["date_collecte"].max())
print("  Lignes:", len(train_data))


print("\nTEST")
print("  Début :", test_data["date_collecte"].min())
print("  Fin   :", test_data["date_collecte"].max())
print("  Lignes:", len(test_data))


print("\nVérification :")


if train_data["date_collecte"].max() < test_data["date_collecte"].min():
    print("✅ Séparation temporelle correcte")
else:
    print("❌ ERREUR : chevauchement temporel détecté")

In [ ]:
# ============================================================
# CELLULE 4 — SÉPARATION TEMPORELLE TRAIN / TEST
# ============================================================


# Vérification et préparation de la date
data["date_collecte"] = pd.to_datetime(data["date_collecte"])


# Même frontière temporelle que celle corrigée dans le pipeline
DATE_CUTOFF = pd.Timestamp("2025-03-15")


train_data = data[data["date_collecte"] < DATE_CUTOFF].copy()
test_data = data[data["date_collecte"] >= DATE_CUTOFF].copy()


print("SÉPARATION TEMPORELLE")
print("-" * 50)


print("TRAIN")
print("  Début :", train_data["date_collecte"].min())
print("  Fin   :", train_data["date_collecte"].max())
print("  Lignes:", len(train_data))


print("\nTEST")
print("  Début :", test_data["date_collecte"].min())
print("  Fin   :", test_data["date_collecte"].max())
print("  Lignes:", len(test_data))


print("\nVérification :")


if train_data["date_collecte"].max() < test_data["date_collecte"].min():
    print("✅ Séparation temporelle correcte")
else:
    print("❌ ERREUR : chevauchement temporel détecté")

# ============================================================
# CELLULE 5 — CONSTRUCTION DES X_train / X_test
# ============================================================


datasets = {}


y_train = train_data[TARGET]
y_test = test_data[TARGET]


for nom, features in configs.items():

    X_train = train_data[features].copy()
    X_test = test_data[features].copy()

    datasets[nom] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    print(f"\n{nom}")
    print("  X_train :", X_train.shape)
    print("  X_test  :", X_test.shape)
    print("  y_train :", y_train.shape)
    print("  y_test  :", y_test.shape)

In [ ]:
# ============================================================
# CELLULE 5 — CONSTRUCTION DES X_train / X_test
# ============================================================


datasets = {}


y_train = train_data[TARGET]
y_test = test_data[TARGET]


for nom, features in configs.items():

    X_train = train_data[features].copy()
    X_test = test_data[features].copy()

    datasets[nom] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    print(f"\n{nom}")
    print("  X_train :", X_train.shape)
    print("  X_test  :", X_test.shape)
    print("  y_train :", y_train.shape)
    print("  y_test  :", y_test.shape)

# ============================================================
# CELLULE 6 — CONTRÔLES AVANT ENTRAÎNEMENT
# ============================================================


for nom, d in datasets.items():

    X_train = d["X_train"]
    X_test = d["X_test"]

    print(f"\n{'='*60}")
    print(nom)

    print("NaN TRAIN :", X_train.isna().sum().sum())
    print("NaN TEST  :", X_test.isna().sum().sum())

    print("Inf TRAIN :", np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())
    print("Inf TEST  :", np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())

    print("Colonnes identiques :",
          list(X_train.columns) == list(X_test.columns))

    if (
        X_train.isna().sum().sum() == 0
        and X_test.isna().sum().sum() == 0
        and list(X_train.columns) == list(X_test.columns)
    ):
        print("✅ Configuration prête")
    else:
        print("❌ Vérification nécessaire")

In [ ]:
# ============================================================
# CELLULE 6 — CONTRÔLES AVANT ENTRAÎNEMENT
# ============================================================


for nom, d in datasets.items():

    X_train = d["X_train"]
    X_test = d["X_test"]

    print(f"\n{'='*60}")
    print(nom)

    print("NaN TRAIN :", X_train.isna().sum().sum())
    print("NaN TEST  :", X_test.isna().sum().sum())

    print("Inf TRAIN :", np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())
    print("Inf TEST  :", np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())

    print("Colonnes identiques :",
          list(X_train.columns) == list(X_test.columns))

    if (
        X_train.isna().sum().sum() == 0
        and X_test.isna().sum().sum() == 0
        and list(X_train.columns) == list(X_test.columns)
    ):
        print("✅ Configuration prête")
    else:
        print("❌ Vérification nécessaire")

In [ ]:
# ============================================================
# ÉTAPE 7B — FEATURES CANDIDATES
# ============================================================


features_candidates = [
    "fillRate",
    "fillRate_t_minus_1",
    "fillRate_t_minus_2",
    "delta_fillRate_24h",
    "rolling_mean_3d",
    "jours_depuis_derniere_collecte",
    "capacity_m3",
    "poids_kg",
    "densite_population_hab_km2",
    "nb_signalements_citoyens",
    "nb_plaintes",
    "nb_precollecteurs_dispo",
    "pression_citoyenne",
    "pression_collecte",
    "charge_precollecteur",
    "indice_saturation",
    "risque_debordement",
    "rendement_remplissage"
]


print("Variables candidates :")


for feature in features_candidates:
    if feature in data.columns:
        print(f"✅ {feature}")
    else:
        print(f"❌ {feature}")

In [ ]:
# ============================================================
# ÉTAPE 7B-1 — NOUVELLES FEATURES
# ============================================================


# 1. Variation du remplissage sur 3 jours
data["delta_fillRate_3d"] = (
    data["fillRate"] - data["fillRate_t_minus_2"]
)


# 2. Accélération du remplissage
# Compare la variation actuelle avec la variation précédente
data["delta_fillRate_precedente"] = (
    data["fillRate_t_minus_1"] - data["fillRate_t_minus_2"]
)


data["acceleration_fillRate"] = (
    data["delta_fillRate_24h"]
    - data["delta_fillRate_precedente"]
)


# 3. Interaction remplissage × délai depuis dernière collecte
data["fillRate_x_jours_collecte"] = (
    data["fillRate"] * data["jours_depuis_derniere_collecte"]
)


print("✅ 3 nouvelles features créées :")
print("- delta_fillRate_3d")
print("- acceleration_fillRate")
print("- fillRate_x_jours_collecte")

In [ ]:
# ============================================================
# CONTRÔLE ANTI-FUITE DES NOUVELLES FEATURES
# ============================================================


nouvelles_features = [
    "delta_fillRate_3d",
    "delta_fillRate_precedente",
    "acceleration_fillRate",
    "fillRate_x_jours_collecte"
]


print("Controle des nouvelles features :
")


for feature in nouvelles_features:
    print(f"{feature} : OK")


# Vérification qu'aucune nouvelle feature n'utilise directement la cible
for feature in nouvelles_features:
    if TARGET in feature:
        print(f"❌ ATTENTION : {feature} semble liée à la cible")
    else:
        print(f"✅ {feature} — pas de référence directe à la cible")

In [ ]:
# ============================================================
# CONTRÔLE DES VALEURS
# ============================================================


for feature in nouvelles_features:

    nan_count = data[feature].isna().sum()
    inf_count = np.isinf(data[feature]).sum()

    print(
        f"{feature} | "
        f"NaN = {nan_count} | "
        f"Inf = {inf_count}"
    )